In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import time
import os

from matplotlib.pyplot import color_sequences

# Task1

In [2]:
registration = [
    {"username": "jdoe88", "email": "john.doe@example.com", "age": 29, "signup_date": "2024-01-15"},
    {"username": "tech_wizard", "email": "wizard@domain.org", "age": 34, "signup_date": "2024-01-16"},
    {"username": "nature_lover", "email": "green.thumb@forest.net", "age": 22, "signup_date": "2024-01-17"},
    {"username": "pixel_artist", "email": "draw.it@canvas.com", "age": 41, "signup_date": "2024-01-18"},
    {"username": "travel_bug", "email": "miles@horizon.com", "age": 27, "signup_date": "2024-01-19"},
    {"username": "glitch_user", "email": "oops-no-at-sign.com", "age": 31, "signup_date": "2024-01-20"},
    {"username": "retro_gamer", "email": "joystick@arcade.io", "age": "twenty-five", "signup_date": "2024-01-21"},
    {"username": "", "email": "anonymous@privacy.com", "age": 50, "signup_date": "2024-01-22"}
]

1) This represents User Input. These fields are typically collected via a web form or
mobile app registration screen.
2) Because humans are involved, you can expect "dirty" data
3) Before this hits a pipeline, you should apply Schema Validation to ensure types are correct (e.g., age must be an int). You would also need Regex Validation for the email format and Outlier Detection to ensure the age falls within a realistic range

In [3]:
logs = [
    {"timestamp": "2026-03-02 08:00:01", "service": "AuthService", "level": "INFO", "message": "User login successful for session_id: 8842"},
    {"timestamp": "2026-03-02 08:05:12", "service": "PaymentGateway", "level": "INFO", "message": "Transaction started for order_id: TXN_9901"},
    {"timestamp": "2026-03-02 08:05:15", "service": "PaymentGateway", "level": "ERROR", "message": "Connection timeout while reaching bank API"},
    {"timestamp": "2026-03-02 08:10:45", "service": "InventoryManager", "level": "WARNING", "message": "Stock level for item 'SKU-44' is below 10%"},
    {"timestamp": "2026-03-02 08:12:00", "service": "AuthService", "level": "WARNING", "message": "Multiple failed login attempts detected for user 'jdoe88'"},
    {"timestamp": "2026-03-02 08:15:30", "service": "PaymentGateway", "level": "INFO", "message": "Retry successful for order_id: TXN_9901"},
    {"timestamp": "2026-03-02 08:20:10", "service": "InventoryManager", "level": "INFO", "message": "Inventory refresh completed"},
    {"timestamp": "2026-03-02 08:22:55", "service": "AuthService", "level": "ERROR", "message": "Database connection pool exhausted"},
    {"timestamp": "2026-03-02 08:25:00", "service": "InventoryManager", "level": "INFO", "message": "New product 'SKU-99' added to database"},
    {"timestamp": "2026-03-02 08:30:15", "service": "PaymentGateway", "level": "INFO", "message": "Monthly settlement report generated"}
]

1) This represent System_generated datas. These datas collected via system generated logs.
2) The primary issues here are Unstructured text (messages can change if a developer updates the code)
3) Normalize timestamp and apply categorical data for level and servic fields

In [4]:
products = [
    {"sku": "LAP-X1-BLK", "product_name": "Zenith Laptop 15in", "quantity_in_stock": 42, "warehouse_location": "Aisle-3-Shelf-B", "last_updated": "2026-03-01 09:15:00"},
    {"sku": "MOU-WL-404", "product_name": "Silent Optical Mouse", "quantity_in_stock": 156, "warehouse_location": "Aisle-1-Shelf-A", "last_updated": "2026-03-01 10:20:45"},
    {"sku": "KBD-MECH-RGB", "product_name": "Mechanical RGB Keyboard", "quantity_in_stock": 8, "warehouse_location": "Aisle-1-Shelf-C", "last_updated": "2026-03-01 11:05:12"},
    {"sku": "MON-4K-27", "product_name": "UltraView 4K Monitor", "quantity_in_stock": 25, "warehouse_location": "Aisle-5-Shelf-D", "last_updated": "2026-03-02 08:30:00"},
    {"sku": "USB-C-HUB-6", "product_name": "6-in-1 USB-C Hub", "quantity_in_stock": 310, "warehouse_location": "Aisle-2-Shelf-A", "last_updated": "2026-03-02 09:45:30"},
    {"sku": "HDP-NC-PRO", "product_name": "Noise Cancelling Headphones", "quantity_in_stock": 12, "warehouse_location": "Aisle-4-Shelf-B", "last_updated": "2026-03-02 11:55:10"}
]

1) This represents an Internal Database. It is the "source of truth" for physical assets
2) Duplicate SKUs due to system mergers
3) For a demand forecasting pipeline, you would need to check for Negative Values and duplicate values for sku

In [5]:
def validate_registration(entries):
    for entry in entries:
        valid_entries = []
        invalid_entries = []

        username = entry.get("username", "")
        email = entry.get("email", "")
        age_raw = entry.get("age")
        sign_up_date =entry.get("sign_up_date")

        is_valid = True

        if not username or str(username).strip() == "":
            is_valid = False
        elif not email or "@" not in email:
            is_valid = False
        elif not sign_up_date == time.strptime(sign_up_date, "%Y-%m-%d %H:%M:%S"):
            is_valid = False
        try:
            age_int = int(age_raw)
            if age_int <= 0:
                is_valid = False
        except (ValueError, TypeError):
            is_valid = False

        if is_valid:
            valid_entries.append(entry)
        else:
            invalid_entries.append(entry)

    return valid_entries, invalid_entries


# Task2

In [6]:
np.random.seed(11)

n_rows = 500

data = {
    "ride_id": np.arange(1, n_rows + 1),
    "driver_id": np.random.randint(1000, 1051, size=n_rows),
    "distance_km": np.round(np.random.uniform(1.0, 30.0, size=n_rows), 2),
    "fare_usd": np.round(np.random.uniform(5.0, 75.0, size=n_rows), 2),
    "ride_type": np.random.choice(["standard", "premium", "pool"], size=n_rows),
    "city": np.random.choice(["Berlin", "Seoul", "Nairobi", "Toronto", "Lima"], size=n_rows),
    "timestamp": pd.date_range(start="2025-01-01", periods=n_rows, freq="min")
}

rides = pd.DataFrame(data)

rides.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ride_id      500 non-null    int64         
 1   driver_id    500 non-null    int64         
 2   distance_km  500 non-null    float64       
 3   fare_usd     500 non-null    float64       
 4   ride_type    500 non-null    str           
 5   city         500 non-null    str           
 6   timestamp    500 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(2), int64(2), str(2)
memory usage: 33.3 KB


In [7]:
rides.to_csv("rides.csv", index=False)
rides.to_parquet("rides.parquet")
rides.to_json("rides.json",orient="records",lines=True)

/tmp/ipykernel_88106/347899426.py:3: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  rides.to_json("rides.json",orient="records",lines=True)


In [8]:
files = ["rides.csv", "rides.json", "rides.parquet"]
sizes = {f: os.path.getsize(f) / 1024 for f in files}
sizes

{'rides.csv': 26.4853515625,
 'rides.json': 64.5087890625,
 'rides.parquet': 17.3623046875}

In [9]:
ratio = sizes["rides.csv"] / sizes["rides.parquet"]
ratio

1.5254513752179537

In [10]:
df_csv =pd.read_csv("rides.csv")
df_csv.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ride_id      500 non-null    int64  
 1   driver_id    500 non-null    int64  
 2   distance_km  500 non-null    float64
 3   fare_usd     500 non-null    float64
 4   ride_type    500 non-null    str    
 5   city         500 non-null    str    
 6   timestamp    500 non-null    str    
dtypes: float64(2), int64(2), str(3)
memory usage: 42.6 KB


In [11]:
df_json = pd.read_json("rides.json",lines=True)
df_json.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ride_id      500 non-null    int64         
 1   driver_id    500 non-null    int64         
 2   distance_km  500 non-null    float64       
 3   fare_usd     500 non-null    float64       
 4   ride_type    500 non-null    str           
 5   city         500 non-null    str           
 6   timestamp    500 non-null    datetime64[ms]
dtypes: datetime64[ms](1), float64(2), int64(2), str(2)
memory usage: 33.3 KB


In [12]:
df_parquet = pd.read_parquet("rides.parquet")
df_parquet.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ride_id      500 non-null    int64         
 1   driver_id    500 non-null    int64         
 2   distance_km  500 non-null    float64       
 3   fare_usd     500 non-null    float64       
 4   ride_type    500 non-null    str           
 5   city         500 non-null    str           
 6   timestamp    500 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(2), int64(2), str(2)
memory usage: 33.3 KB


1) Parquet stores the schema (metadata) directly within the file. When you reload a Parquet file, an integer stays an int64 and a timestamp stays a datetime64[us]. JSON (with lines=True) also does a great job with dates in pandas, but CSV fails completely—it treats everything as strings or numbers, forcing you to manually use parse_dates=['timestamp'].

2) Parquet was significantly smaller (roughly 1.5x smaller than CSV in this 500-row sample).

3) Use csv when you must open data in excel or share data someone who dont use python/code.
But for ML pipelines parquet is best choice when perfomance and type safety are priorty.
JSON mostly used for web APIs

# Task3

In [13]:
np.random.seed(11)

cols = 20
rows =200000

data = np.random.randn(rows, cols)

colum_names = ([f'feature_{i}' for i in range(cols)])

df = pd.DataFrame(data, columns=colum_names)

In [14]:
%time df["feature_0"].mean()

CPU times: user 2.38 ms, sys: 71 μs, total: 2.45 ms
Wall time: 1.46 ms


np.float64(0.0026301780655443537)

In [15]:
%%timeit
sum = 0
for i in  range(10000+1):
     sum += df['feature_0'].iloc[i]

149 ms ± 4.57 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [16]:
arr = df.to_numpy()
%time arr[:, 0].mean()

CPU times: user 487 μs, sys: 0 ns, total: 487 μs
Wall time: 340 μs


np.float64(0.0026301780655443537)

In [17]:
%%time
sum_val = 0
for i in range(10001):
    sum_val += arr[i, 0]
sum_val

CPU times: user 1.72 ms, sys: 0 ns, total: 1.72 ms
Wall time: 1.73 ms


np.float64(110.00634802625697)

1) Column access in pandas is much faster than row iteration because pandas DataFrames are stored as column-major internally (often backed by homogeneous NumPy arrays). Accessing a column is a continuous memory read, while iterating row-by-row involves significant overhead from pandas index checking and Series object creation for each row.
2) The gap narrows when using NumPy because NumPy arrays (in C-contiguous order) are row-major by default. Iterating through rows in a NumPy array accesses memory more sequentially and avoids all pandas-specific overhead.
3) This tells us that pandas optimizes for column-oriented operations, whereas default NumPy arrays are row-major, making row-wise operations much faster in NumPy than in pandas.

# Task4

In [18]:
np.random.seed(42)
n_rows_t4 = 100000
df_t4 = pd.DataFrame({
    "col1": np.random.rand(n_rows_t4),
    "col2": np.random.rand(n_rows_t4),
    "col3": np.random.rand(n_rows_t4),
    "col4": np.random.rand(n_rows_t4),
    "col5": np.random.rand(n_rows_t4),
    "category": np.random.choice(["alpha", "beta", "gamma"], size=n_rows_t4)
})
df_t4.head()

,col1,col2,col3,col4,col5,category
0,0.374540,0.580779,0.282588,0.157054,0.888953,alpha
1,0.950714,0.526972,0.458677,0.095509,0.315713,gamma
2,0.731994,0.351037,0.099215,0.137939,0.051910,beta
3,0.598658,0.493213,0.446837,0.473489,0.919744,alpha
4,0.156019,0.365097,0.203081,0.884534,0.436261,gamma


In [19]:
df_t4.to_csv("task4.csv", index=False)
df_t4.to_parquet("task4.parquet")

sizes_t4 = {
    "CSV": os.path.getsize("task4.csv") / 1024,
    "Parquet": os.path.getsize("task4.parquet") / 1024
}
sizes_t4

{'CSV': 9962.3642578125, 'Parquet': 4926.9697265625}

In [20]:
df_csv_t4 = pd.read_csv("task4.csv")
df_csv_t4.dtypes

col1        float64
col2        float64
col3        float64
col4        float64
col5        float64
category        str
dtype: object

In [21]:
df_parq_t4 = pd.read_parquet("task4.parquet")
df_parq_t4.dtypes

col1        float64
col2        float64
col3        float64
col4        float64
col5        float64
category        str
dtype: object

When reading from CSV, the string column `category` is loaded as an `object` dtype natively because CSV is just text and does not strictly enforce or store binary type metadata for strings.
When reading from Parquet, depending on the backend, the `category` column can retain string/object types efficiently, and all numeric types are strictly preserved since Parquet stores explicit schema and format metadata.

In [22]:
%time df_csv_read = pd.read_csv("task4.csv")

CPU times: user 61.8 ms, sys: 9.88 ms, total: 71.7 ms
Wall time: 71 ms


In [23]:
%time df_parq_read = pd.read_parquet("task4.parquet")

CPU times: user 10.1 ms, sys: 9.93 ms, total: 20 ms
Wall time: 6.52 ms


In [24]:
%time df_parq_cat = pd.read_parquet("task4.parquet", columns=["category"])

CPU times: user 4.53 ms, sys: 938 μs, total: 5.47 ms
Wall time: 4.01 ms


Selective reads are possible with Parquet because it is a column-major binary format. Data for each column is stored contiguously, and file metadata indicates exactly where each column starts and ends. Therefore, the engine can jump directly to the `category` column and read only its bytes. CSV, however, is a row-major text format; the parser must scan through every line entirely to find and extract the specific column's values, making selective reading just as costly as reading the full file.

# Task5

In [25]:
flat_data = [
    {"title": "Book A", "author": "Author 1", "format": "Paperback", "publisher_name": "Pub X", "publisher_country": "USA", "price": 10.99},
    {"title": "Book A", "author": "Author 1", "format": "E-book", "publisher_name": "Pub X", "publisher_country": "USA", "price": 8.99},
    {"title": "Book B", "author": "Author 2", "format": "Paperback", "publisher_name": "Pub Y", "publisher_country": "UK", "price": 12.50},
    {"title": "Book B", "author": "Author 2", "format": "E-book", "publisher_name": "Pub Y", "publisher_country": "UK", "price": 9.99},
    {"title": "Book C", "author": "Author 1", "format": "Paperback", "publisher_name": "Pub X", "publisher_country": "USA", "price": 15.00},
    {"title": "Book D", "author": "Author 3", "format": "E-book", "publisher_name": "Pub Z", "publisher_country": "Canada", "price": 7.50},
    {"title": "Book E", "author": "Author 4", "format": "Paperback", "publisher_name": "Pub X", "publisher_country": "USA", "price": 14.99},
    {"title": "Book F", "author": "Author 5", "format": "Paperback", "publisher_name": "Pub Y", "publisher_country": "UK", "price": 11.20},
    {"title": "Book G", "author": "Author 6", "format": "E-book", "publisher_name": "Pub Z", "publisher_country": "Canada", "price": 6.99},
    {"title": "Book H", "author": "Author 2", "format": "Paperback", "publisher_name": "Pub Y", "publisher_country": "UK", "price": 13.99},
    {"title": "Book I", "author": "Author 7", "format": "E-book", "publisher_name": "Pub X", "publisher_country": "USA", "price": 5.99},
    {"title": "Book J", "author": "Author 8", "format": "Paperback", "publisher_name": "Pub Z", "publisher_country": "Canada", "price": 10.00}
]
df_flat = pd.DataFrame(flat_data)
df_flat.head()

,title,author,format,publisher_name,publisher_country,price
0,Book A,Author 1,Paperback,Pub X,USA,10.99
1,Book A,Author 1,E-book,Pub X,USA,8.99
2,Book B,Author 2,Paperback,Pub Y,UK,12.50
3,Book B,Author 2,E-book,Pub Y,UK,9.99
4,Book C,Author 1,Paperback,Pub X,USA,15.00


In [26]:
publishers_data = [
    {"publisher_id": 1, "publisher_name": "Pub X", "publisher_country": "USA"},
    {"publisher_id": 2, "publisher_name": "Pub Y", "publisher_country": "UK"},
    {"publisher_id": 3, "publisher_name": "Pub Z", "publisher_country": "Canada"}
]
df_publishers = pd.DataFrame(publishers_data)

books_data = [
    {"title": "Book A", "author": "Author 1", "format": "Paperback", "publisher_id": 1, "price": 10.99},
    {"title": "Book A", "author": "Author 1", "format": "E-book", "publisher_id": 1, "price": 8.99},
    {"title": "Book B", "author": "Author 2", "format": "Paperback", "publisher_id": 2, "price": 12.50},
    {"title": "Book B", "author": "Author 2", "format": "E-book", "publisher_id": 2, "price": 9.99},
    {"title": "Book C", "author": "Author 1", "format": "Paperback", "publisher_id": 1, "price": 15.00},
    {"title": "Book D", "author": "Author 3", "format": "E-book", "publisher_id": 3, "price": 7.50},
    {"title": "Book E", "author": "Author 4", "format": "Paperback", "publisher_id": 1, "price": 14.99},
    {"title": "Book F", "author": "Author 5", "format": "Paperback", "publisher_id": 2, "price": 11.20},
    {"title": "Book G", "author": "Author 6", "format": "E-book", "publisher_id": 3, "price": 6.99},
    {"title": "Book H", "author": "Author 2", "format": "Paperback", "publisher_id": 2, "price": 13.99},
    {"title": "Book I", "author": "Author 7", "format": "E-book", "publisher_id": 1, "price": 5.99},
    {"title": "Book J", "author": "Author 8", "format": "Paperback", "publisher_id": 3, "price": 10.00}
]
df_books = pd.DataFrame(books_data)

df_joined = pd.merge(df_books, df_publishers, on="publisher_id").drop(columns=["publisher_id"])
df_joined = df_joined[["title", "author", "format", "publisher_name", "publisher_country", "price"]]
df_joined.head()

,title,author,format,publisher_name,publisher_country,price
0,Book A,Author 1,Paperback,Pub X,USA,10.99
1,Book A,Author 1,E-book,Pub X,USA,8.99
2,Book B,Author 2,Paperback,Pub Y,UK,12.50
3,Book B,Author 2,E-book,Pub Y,UK,9.99
4,Book C,Author 1,Paperback,Pub X,USA,15.00


In [27]:
document_data = [
    {
        "title": "Book A", "author": "Author 1",
        "publisher": {"name": "Pub X", "country": "USA"},
        "editions": [{"format": "Paperback", "price": 10.99}, {"format": "E-book", "price": 8.99}]
    },
    {
        "title": "Book B", "author": "Author 2",
        "publisher": {"name": "Pub Y", "country": "UK"},
        "editions": [{"format": "Paperback", "price": 12.50}, {"format": "E-book", "price": 9.99}]
    },
    {
        "title": "Book C", "author": "Author 1",
        "publisher": {"name": "Pub X", "country": "USA"},
        "editions": [{"format": "Paperback", "price": 15.00}]
    },
    {
        "title": "Book D", "author": "Author 3",
        "publisher": {"name": "Pub Z", "country": "Canada"},
        "editions": [{"format": "E-book", "price": 7.50}]
    },
    {
        "title": "Book E", "author": "Author 4",
        "publisher": {"name": "Pub X", "country": "USA"},
        "editions": [{"format": "Paperback", "price": 14.99}]
    },
    {
        "title": "Book F", "author": "Author 5",
        "publisher": {"name": "Pub Y", "country": "UK"},
        "editions": [{"format": "Paperback", "price": 11.20}]
    },
    {
        "title": "Book G", "author": "Author 6",
        "publisher": {"name": "Pub Z", "country": "Canada"},
        "editions": [{"format": "E-book", "price": 6.99}]
    },
    {
        "title": "Book H", "author": "Author 2",
        "publisher": {"name": "Pub Y", "country": "UK"},
        "editions": [{"format": "Paperback", "price": 13.99}]
    },
    {
        "title": "Book I", "author": "Author 7",
        "publisher": {"name": "Pub X", "country": "USA"},
        "editions": [{"format": "E-book", "price": 5.99}]
    },
    {
        "title": "Book J", "author": "Author 8",
        "publisher": {"name": "Pub Z", "country": "Canada"},
        "editions": [{"format": "Paperback", "price": 10.00}]
    }
]
document_data[0]

{'title': 'Book A',
 'author': 'Author 1',
 'publisher': {'name': 'Pub X', 'country': 'USA'},
 'editions': [{'format': 'Paperback', 'price': 10.99},
  {'format': 'E-book', 'price': 8.99}]}

In [28]:
# Flat Representation
author1_books_flat = df_flat[df_flat['author'] == 'Author 1']['title'].unique()
avg_price_flat = df_flat['price'].mean()

rows_to_change_flat = (df_flat['publisher_name'] == 'Pub X').sum()
df_flat.loc[df_flat['publisher_name'] == 'Pub X', 'publisher_country'] = 'Australia'

(author1_books_flat, avg_price_flat, rows_to_change_flat)

(<ArrowStringArray>
 ['Book A', 'Book C']
 Length: 2, dtype: str,
 np.float64(10.6775),
 np.int64(5))

In [29]:
# Normalized Representation
author1_books_norm = df_books[df_books['author'] == 'Author 1']['title'].unique()
avg_price_norm = df_books['price'].mean()

rows_to_change_norm = (df_publishers['publisher_name'] == 'Pub X').sum()
df_publishers.loc[df_publishers['publisher_name'] == 'Pub X', 'publisher_country'] = 'Australia'

(author1_books_norm, avg_price_norm, rows_to_change_norm)

(<ArrowStringArray>
 ['Book A', 'Book C']
 Length: 2, dtype: str,
 np.float64(10.6775),
 np.int64(1))

In [30]:
# Document Representation
author1_books_doc = [doc['title'] for doc in document_data if doc['author'] == 'Author 1']
all_prices = [edition['price'] for doc in document_data for edition in doc['editions']]
avg_price_doc = sum(all_prices) / len(all_prices)

docs_to_change = sum(1 for doc in document_data if doc['publisher']['name'] == 'Pub X')
for doc in document_data:
    if 'publisher' in doc and doc['publisher']['name'] == 'Pub X':
        doc['publisher']['country'] = 'Australia'

(author1_books_doc, avg_price_doc, docs_to_change)

(['Book A', 'Book C'], 10.6775, 4)

### Summary
1. **Finding all books by a specific author**: Easiest in the Flat and Normalized formats using concise pandas filtering (`df[df['author'] == '...']`). In the Document format, it requires a loop or list comprehension.
2. **Finding the average price**: Easiest in the Flat and Normalized formats where we can just call `.mean()` on the price column directly. In the Document format, it requires nested looping to extract all prices across editions.
3. **Updating a publisher's country**: Easiest and safest in the Normalized format, as exactly **1** record in the separate publishers table needs to change. In both Flat and Document formats, we must update multiple records/documents, creating the risk of data anomalies.

**Representation Choices:**
- **If publisher information changes frequently**, choose the **Normalized** representation. It avoids data redundancy and update anomalies.
- **If each book is always accessed as a standalone record** (e.g., retrieving a book page in a web app with all its details), choose the **Document** representation. It provides all relevant information without needing expensive database joins.